# Project Write Up

## 1. Introduction

A core workflow in private credit involves distilling lengthy transaction documents into concise materials for investment committee (IC) review. These materials (typically a short written memo or the opening slides of an IC deck) summarize key terms, strengths and risks, and high-level rationale for the investment. The underlying information is usually contained in credit agreements and related legal documents that can be several hundred pages long, written in dense legal language, and require substantial manual effort to parse. Analysts must identify terms that are commercially relevant and interpret them through the lens of a credit investor.

While modern AI tools have begun to support well-defined financial tasks (e.g., LBO modeling) or legal workflows, few systems address this intersection: understanding legal documentation while reasoning like an investor about what constitutes a strong or weak credit opportunity. This project explores whether LLMs can meaningfully automate this workflow and how their performance can be evaluated systematically.  

The project has two primary goals:
1. Create a **memo generation pipeline** that ingests credit agreements scraped from public filings, extracts meaningful information, and generates a structured investment memo. Due to context limits, the output is scoped to the first one to two pages of an IC deck - an executive summary highlighting transaction details, key terms, and a concise assessment of strengths and risks.
2. Develop an **evaluation harness** to measure LLM performance across models (e.g., GPT-5, Claude Opus 4, Gemini 2.5 Pro) and across prompting strategies. The harness applies consistent metrics to assess whether techniques such as few-shot prompting, iterative refinement, chain-of-thought prompting, and more lead to systematic improvements.

Several avenues of model customization were initially considered, particularly supervised fine-tuning and reinforcement learning-based approaches, but were deprioritized for several reasons. Fine-tuning can yield strong results for narrow, well-labeled domains, but several limitations made it unsuitable as a first step: 
- **Lack of labeled data**: Although it was fairly straightforward to scrape a large corpus of credit agreements, creating a high-quality dataset of paired inputs and outputs (credit agreements → investment memos) would have required substantial manual effort.
- **Risk of overfitting**: With limited labeled data, a fine-tuned model might simply mimic the style of a small “gold standard” set rather than truly “learn” the investment rationale and be able to generalize across diverse credit agreements. A fine tuned model also needs to be re-tuned as new data is available, and / or when a new model is released.
- **Compute constraints**: Training (even at small scale) was not feasible within available resources.
- **Premature specialization**: Before customizing a model, it felt more valuable to build an end-to-end pipeline and establish objective criteria for evaluating quality.

Reinforcement learning presented similar obstacles (lack of reward signals, difficulty defining an environment, significant compute demands…). Given these limitations, a prompt engineering approach offered the highest return on time and allowed rapid iteration towards a working system.


The project therefore centers on building a **practical, evaluation-driven system**. The focus is on designing an end-to-end pipeline that works on real financial documents, establishing measurable criteria for high-quality memo generation, understanding how prompt design influences output quality, and identifying where current LLMs fall short when interpreting and reasoning over legal credit documents. This evaluation-first foundation hopefully provides a realistic view of current model capabilities and creates a clear path for future extensions.

## 2. Dataset Build

### Data Acquisition

This project uses public credit agreements filed on the SEC’s EDGAR system by large public companies (2023 onward) as inputs for memo generation. These documents are well suited because they are publicly accessible, contain consistent legal and financial terms (e.g., interest rates, maturities, covenants), and follow broadly standard credit-agreement structures.

Data was retrieved using [EDGAR Full-Text Search queries](https://www.sec.gov/edgar/search/#/q=%2522Credit%2520Agreement%2522%2520AND%2520EX-10.1) for filings containing “Credit Agreement” AND “EX-10.1.” This reliably retrieves credit agreement exhibits across issuers. A simple web scraper (see below) extracted document URLs and downloaded the corresponding HTML files. This process produced 500 agreements (499 unique), providing a sufficiently large corpus for experimentation without paid data sources. For downstream processing, documents were stored in JSONL format with source URLs and plain-text content.

### Data Cleaning

Although EDGAR documents are generally high quality, they contain artifacts (headers, footers, disclaimers,...) that can hinder extraction. A preprocessing pipeline transformed raw filings into clean, model-ready text. The pipeline reads URLs; downloads each page; converts the HTML to normalized plain text (removing scripts, styles and artifacts); writes one JSON line per document with source URL and text.

### Train-Test Split

A deterministic URL-based hash function partitioned documents into training and evaluation sets, ensuring reproducibility and preventing leakage. Fifty documents were used for training, with the remaining 449 held out for testing – after exploratory runs, iterating on more than 50 documents proved too costly and time intensive.

## 3. Evaluation Framework

### Chosen Benchmark Model

For the scope of this project, a single model needed to be chosen to iterate on, so as to explore whether prompt optimizing techniques could improve that given model. While this choice could have been made arbitrarily, one of three frontier models was picked (Gemini 2.5 pro, Claude Sonnet 4, GPT-5) based on some exploratory runs. This was done by sampling 3 inputs from the training set, and manually inspecting how the three models’ outputs differed - what instructions can the model follow, where does it seem to struggle? This was done prior to building out a full evaluation harness, so as to help inform the evaluation metrics, and ensure that they accurately assessed what seemed to be the main pain points of model performance.

For these exploratory runs, a template was built of what a typical memo should look like, and a thoughtful prompt that an “average” person would spend a few minutes writing was drafted: a brief system preamble outlining what documents to use and how long the output should be *“You are an investment analyst. Using the provided credit agreement and any template references, produce a concise, structured investment memo. Keep the total output under 400 words or 15k tokens”*, followed by a thoughtful prompt providing more detailed information on the task: 

> <div style="font-size:70%; color:#666;">
>
> You are a Private Credit Analyst at an investment fund. Using the information contained in the attached credit agreement, draft a professional investment memorandum structured in three sections:
>
> 1. Executive Summary: Provide a concise overview that includes:
> - Date of the agreement  
> - Borrower / Company overview  
> - Brief description of the transaction (type, structure, counterparties)  
> - Purpose of the financing  
> - Brief company background and context
>
> 2. Investment Highlights & Risks: Present clear, bullet-pointed analysis from the perspective of an investor:
> - Key strengths / credit positives  
> - Principal risks and mitigating factors
>
> 3. Key Deal Terms Table: Include a well-formatted table listing:
> - Deal size  
> - Deal price  
> - Interest rate (and type, if applicable)  
> - Maturity date  
> - Payment frequency  
> - Key covenants or financial maintenance terms
>
> Instructions:
> - Use only factual information explicitly found in the attached credit agreement.  
> - If specific data points are not provided, write "N/A"—do not infer or fabricate details.  
> - Maintain a clear, concise, and professional tone suitable for an internal investment committee memo.  
> - Align the structure, level of detail, and tone with the attached template memo for reference.
>
> </div>


While this prompt was not perfectly optimized and included some redundancies, it was thought of as a good benchmark of what an individual might prompt into a chatbot, and chosen as a baseline.

Based on the exploratory 3-input run, it seemed that the general structure and tone was respected by all 3 models. While the content was largely similar across models for all 3 inputs, Claude performed significantly better on one input in identifying some risks that GPT-5 and Gemini 2.5 pro did not find (they seemed to rely solely on highlights that were more explicitly written and did not necessarily make sense, for example gpt-5 named a highlight “execution supported by signatures” which is not a material investment highlight). Gemini 2.5 pro and GPT-5 also had a handful of additional hallucinations compared to Claude, though these were likely not statistically significant. 

Claude Sonnet 4 was therefore chosen as the baseline model to iterate on. This was largely a high level judgement, and did not reflect a systematic, robust comparison of the three models which was made more difficult by limited compute/time. 

### Chosen Evaluation Metrics

Model performance is typically assessed along three dimensions: latency, cost, and functional correctness. Since this project evaluates prompting strategies on the same three underlying models, latency and cost were not incorporated directly into evaluation scores. They were tracked informally, particularly where prompting variants (e.g., refinement loops, batched vs. non-batched calls) produced noticeable differences, but the core evaluation focused on functional correctness, defined through several dimensions relevant to memo-quality in a private-credit context. Functional correctness in this context was split into four main components:

#### 1. Accuracy

**Definition**: Whether the memo includes any financial terms or statements that do not appear in the underlying credit agreement.

**Implementation**: A three-model LLM consensus (GPT-5, Claude-Sonnet-4, Gemini-2.5-Pro) answers a targeted yes/no question for each memo. The accuracy score is the share of judges answering “no hallucinations present.” A semantic or rule-based matching approach was initially considered but rejected for several reasons, including:
- Credit agreements vary in formatting/wording (for example, the interest rate could be defined as “interest rate”, “note rate”, or as a “margin” above a set “base rate”...), making it difficult to exhaustively enumerate what “key terms.” could be 
- Semantic-match pipelines are brittle to paraphrasing (e.g., 5.25% vs. 525 bps).

LLM consensus was therefore preferred as it allows for context-specific understanding (distinguishing key economic terms from ancillary text), handles synonyms and reformulations naturally, adapts across heterogeneous document structures, and reduces single-model bias via voting.

#### 2. Completeness

**Definition**: Whether the memo omits any key terms that a credit analyst would reasonably expect to see.


**Implementation**: Same three-model consensus framework as accuracy, with a yes/no question (counted as a score of 1 or 0) asking whether any important information appears to be missing relative to the source document. 

#### 3. Quality

**Definition**: Whether the memo is well-structured, clear, appropriately toned, and has a structure consistent with the template.

**Implementation**: LLM judges score four sub-dimensions. The overall score is the average of the three judges’ scores across these four submetrics.
- Clarity
- Tone 
- Length (concise but complete)
- Structure (alignment with the memo template)

Each sub-metric has its own prompt, for example for tone the prompt was: 

> <div style="font-size:80%; color:#666;">
>
> You are evaluating an investment memo for appropriate professional tone.
>
> MEMO: {memo}
>
> Goal: Assess whether the memo's tone is appropriate for presentation to an investment committee.
>
> Tone Criteria:
> - Professional formality: Language is formal and businesslike, appropriate for executive decision-makers  
> - Objective presentation: Presents information factually without emotional language or hype  
> - Balanced perspective: Acknowledges both strengths and risks without being overly promotional or pessimistic  
> - Financial sophistication: Uses appropriate financial terminology without being overly technical or simplistic  
> - Confidence without arrogance: Presents analysis authoritatively but remains measured
>
> Evaluate the memo's tone on a scale from 0–100:
> - 90–100: Perfectly appropriate for investment committee, professional and balanced  
> - 70–89: Generally appropriate with minor tone issues  
> - 50–69: Somewhat appropriate but has noticeable tone problems  
> - 30–49: Inappropriate tone in multiple sections  
> - 0–29: Significantly inappropriate tone throughout
>
> Output format: Provide ONLY a number from 0–100 as your score.
>
> SCORE: [number]
>
> </div>

#### 4. Consistency

**Definition**: Whether the memo contradicts itself (e.g., listing a term as both a strength and a weakness, or stating different interest rates in different sections).

**Implementation**: Each judge answers a binary yes/no question using a consistency-check prompt focusing on logical coherence within the memo. Measures of stability across repeated runs were considered, including “worst-at-k” (i.e. the worst score among k model runs), but were excluded due to time and compute constraints.

### Evaluation Harness 

To compare prompting strategies systematically, the project required a workflow capable of generating and evaluating many memos efficiently. The main constraints were cost and runtime: each run involved generating dozens of outputs, with each output being evaluated by three different LLM judges using several prompts for each judge. Running evaluations through standard synchronous API calls quickly proved impractical, with initial attempts taking several hours.

To address this, all generation and evaluation steps were migrated to batch API workflows, allowing requests to be parallelized at scale. This reduced typical end-to-end evaluation time for a full run (50 inputs) to a couple of minutes, provided that the evaluation prompts were simple one-turn queries (i.e., no refinement or iterative loops). Batch processing also reduced cost by taking advantage of discounted pricing and minimizing idle compute time.

The evaluation harness follows a consistent five-step pipeline:
1. **Memo Generation**: For a given prompt configuration, memos are generated by a chosen model. All generation requests are submitted as a single batch job for efficiency.
2. **Batch Completion and Retrieval**: The system polls for job completion and downloads all generated memos once the batch finishes.
3. **Multi-Model Evaluation**: Each memo is evaluated independently by three LLM judges using the evaluation prompts defined in Section 3.1. These evaluations are also submitted as (three) batch jobs to parallelize the workload.
4. **Polling and Result Extraction**: Once evaluations complete, results are retrieved and parsed into structured JSONL outputs.
5. **Aggregation and Summary Statistics**: For each memo, metric scores are averaged across judges. At the run level, the harness computes mean, median, minimum, maximum, and standard deviation across all memos and metrics. These aggregates enable comparative analysis across prompting strategies.

This harness provided a repeatable, scalable framework for testing prompt-engineering techniques under consistent conditions. It enabled rapid iteration across many runs and models, making it possible to attempt to isolate the impact of specific prompt changes on memo quality.

### Benchmark Performance

The table below outlines the benchmark performance for the project - the initial baseline prompt fed to Claude Sonnet 4 on the full 50-input training dataset. As outlined below, the baseline (average score) is 61, suggesting moderate performance with definite room for improvement.The large range of scores (from ~27 to ~86) and standard deviation of 15.12 indicated somewhat high inconsistency across samples, though it is unclear if this is driven by heterogeneity in the data quality or in model performance. Tone, clarity and length seemed to be quite strong, while the model seems to struggle with completeness issues (more than half of the model votes across the dataset indicated completeness issues) and with structure. 

<div style="display:flex; gap:20px; align-items:flex-start;">

<table>
  <thead>
    <tr><th colspan="2">Benchmark</th></tr>
  </thead>
  <tbody>
    <tr><td>Total Memos Evaluated</td><td>50</td></tr>
    <tr><td>Mean</td><td>61.28</td></tr>
    <tr><td>Median</td><td>63.58</td></tr>
    <tr><td>Min</td><td>26.85</td></tr>
    <tr><td>Max</td><td>85.77</td></tr>
    <tr><td>Std Dev</td><td>15.12</td></tr>
  </tbody>
</table>

<table>
  <thead>
    <tr><th>Evaluator</th><th>Mean</th><th>Median</th></tr>
  </thead>
  <tbody>
    <tr><td>Claude Sonnet 4</td><td>76.7</td><td>67.7</td></tr>
    <tr><td>GPT-5</td><td>53.3</td><td>45.0</td></tr>
    <tr><td>Gemini 2.5 Pro</td><td>53.8</td><td>45.6</td></tr>
  </tbody>
</table>

<table>
  <thead>
    <tr><th>Metric</th><th>Score</th></tr>
  </thead>
  <tbody>
    <tr><td>Tone</td><td>90.2</td></tr>
    <tr><td>Clarity</td><td>79.4</td></tr>
    <tr><td>Length</td><td>79.1</td></tr>
    <tr><td>Structure</td><td>43.8</td></tr>
    <tr><td>Accuracy (% no hallucinations)</td><td>72.7%</td></tr>
    <tr><td>Completeness (% complete)</td><td>38.7%</td></tr>
    <tr><td>Consistency (% no issues)</td><td>60.7%</td></tr>
  </tbody>
</table>

</div>

Taking a closer look at variability in evaluation scores by evaluator, there seems to be potential self-agreement bias, where Claude’s evaluation of itself ranked much higher both on a mean and a median basis than the other two evaluators. Potential ways to mitigate this could have been to weigh gpt-5 and gemini 2.5 pro’s evaluations more heavily in the overall averaging, but given the small size of the dataset, it wasn’t clear if there was a true bias so equal weighings were kept.

Examining the best scoring and worst scoring outputs for this run reveals that the best scoring output was indeed very high quality, and the worst scoring output was an instance in which all three evaluators agreed on several of the memo’s shortcomings (that were true), leading to very poor scores because the averages were low. In both cases, the ratings generally made sense. This suggests that extreme scores are more reliable, as they reflect full evaluator agreement, and that the “middle range’ scores don’t necessarily reveal much / could just reflect models disagreeing rather than mediocre performance. 

## 4. Prompt Optimizing

With the evaluation harness in place, a variety of prompt optimizing techniques were explored with the aim of improving model performance. These approaches were informed by several guides including those from [OpenAI](https://platform.openai.com/docs/guides/prompt-engineering), [Google](https://cloud.google.com/discover/what-is-prompt-engineering?hl=en), and [Anthropic](https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/overview) on prompt engineering.

### Readily Available Prompt Optimizers

As a first set of experiments, the project evaluated several readily available prompt-optimization tools. These tools are designed to refine an initial “baseline prompt” by applying structured best practices, such as removing ambiguity, tightening instructions, clarifying output formats, and ensuring internal consistency. The goal of this section was to assess whether these off-the-shelf optimizers produced meaningful improvements when applied to this task.

#### Anthropic Prompt Generator

Claude’s built-in prompt generator produced a more explicit and structured version of the baseline prompt, primarily by clarifying document boundaries and enforcing stricter section headers.

Original: *“You are a Private Credit Analyst at an investment fund. Using the information contained in the attached credit agreement, draft a professional investment memorandum structured in three sections:”*

Claude-revised: *“You are acting as a Private Credit Analyst at an investment fund. You will be provided with a credit agreement document and a template memo for reference. Your task is to draft a professional investment memorandum based solely on the information contained in the credit agreement.*<br>*Here is the credit agreement document: <br> <credit_agreement> <br> {{CREDIT_AGREEMENT}}<br> </credit_agreement> <br> Here is the template memo for reference on structure, tone, and level of detail:<br> <template_memo> <br>{{TEMPLATE_MEMO}}<br> </template_memo><br>Your investment memorandum must be structured in exactly three sections:”*

This resulted in modest improvements in the mean, minimum, and maximum scores, and a lower standard deviation. Overall, scores were higher for GPT-5 and Gemini, and lower for Claude, which could be an indication of higher consistency among evaluators/lower self agreement bias. There were slight drops in clarity, tone, length, and structure scores, though it is unclear if this was statistically significant. 

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">63.84</td><td style="text-align:center;">+2.56 pts</td>
    </tr>
    <tr>
      <td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">60.09</td><td style="text-align:center;">−3.49 pts</td>
    </tr>
    <tr>
      <td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">34.75</td><td style="text-align:center;">+7.9 pts</td>
    </tr>
    <tr>
      <td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">93.85</td><td style="text-align:center;">+8.08 pts</td>
    </tr>
    <tr>
      <td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.44</td><td style="text-align:center;">−1.68 pts</td>
    </tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">56.05</td><td style="text-align:center;">+2.72</td>
    </tr>
    <tr>
      <td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">74.99</td><td style="text-align:center;">−1.68</td>
    </tr>
    <tr>
      <td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">60.48</td><td style="text-align:center;">+6.64</td>
    </tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.38</td><td style="text-align:center;">-0.81</td>
    </tr>
    <tr>
      <td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">73.82</td><td style="text-align:center;">-5.55</td>
    </tr>
    <tr>
      <td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">77.24</td><td style="text-align:center;">-1.89</td>
    </tr>
    <tr>
      <td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">43.42</td><td style="text-align:center;">-0.38</td>
    </tr>
    <tr>
      <td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">79.3%</td><td style="text-align:center;">+6.6%</td>
    </tr>
    <tr>
      <td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">40.3%</td><td style="text-align:center;">+1.6%</td>
    </tr>
    <tr>
      <td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">64.7%</td><td style="text-align:center;">+4.0%</td>
    </tr>
  </tbody>
</table>

</div>

#### OpenAI Prompt Optimizer

OpenAI’s prompt optimizer (provided in the OpenAI Cookbook) performs a similar refinement process, explicitly targeting common failure modes: contradictions within instructions, missing or unclear format specifications and inconsistencies between instructions and any included examples/templates – essentially disambiguating the prompt to the maximum extent possible. This optimizer refined the baseline prompt by dividing it into explicit sections with titles (“Role and Objective”, “Instructions”, “Memo Structure”, “Output Format”), and, similarly to Claude, providing even more explicit instructions (for example, replaced “Deal Size” with “Deal Size | Numeric amount with currency”).

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Anthropic)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">63.84</td><td style="text-align:center;">65.72</td><td style="text-align:center;">1.88</td><td style="text-align:center;">4.44</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">60.09</td><td style="text-align:center;">63.99</td><td style="text-align:center;">3.90</td><td style="text-align:center;">0.41</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">34.75</td><td style="text-align:center;">41.77</td><td style="text-align:center;">7.02</td><td style="text-align:center;">14.92</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">93.85</td><td style="text-align:center;">92.52</td><td style="text-align:center;">-1.33</td><td style="text-align:center;">6.75</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.44</td><td style="text-align:center;">11.36</td><td style="text-align:center;">-2.08</td><td style="text-align:center;">-3.76</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Anthropic)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">56.05</td><td style="text-align:center;">57.26</td><td style="text-align:center;">1.21</td><td style="text-align:center;">3.93</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">74.99</td><td style="text-align:center;">77.78</td><td style="text-align:center;">2.79</td><td style="text-align:center;">1.11</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">60.48</td><td style="text-align:center;">62.14</td><td style="text-align:center;">1.66</td><td style="text-align:center;">8.30</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Anthropic)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (OpenAI vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.38</td><td style="text-align:center;">90.00</td><td style="text-align:center;">0.62</td><td style="text-align:center;">-0.19</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">73.82</td><td style="text-align:center;">82.61</td><td style="text-align:center;">8.79</td><td style="text-align:center;">3.24</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">77.24</td><td style="text-align:center;">76.40</td><td style="text-align:center;">-0.84</td><td style="text-align:center;">-2.73</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">43.42</td><td style="text-align:center;">42.58</td><td style="text-align:center;">-0.84</td><td style="text-align:center;">-1.22</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">79.3%</td><td style="text-align:center;">84.0%</td><td style="text-align:center;">4.7%</td><td style="text-align:center;">11.3%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">40.3%</td><td style="text-align:center;">44.7%</td><td style="text-align:center;">4.4%</td><td style="text-align:center;">6.0%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">64.7%</td><td style="text-align:center;">61.3%</td><td style="text-align:center;">-3.4%</td><td style="text-align:center;">0.6%</td></tr>
  </tbody>
</table>

</div>

Generally, this resulted in a sizable improvement compared to the benchmark, with all metrics being generally on par with or better than the benchmark’s performance. Overall, accuracy increased 11 points compared to the benchmark, and almost 5 points compared to the Anthropic optimized prompt. Standard deviation across scores was also much lower, at 11.36 vs. 15.12 in the benchmark. This is considered the best performance so far.

#### Meta-Prompting

Claude was prompted to suggest minimal edits to its own instructions, resulting in a shorter and more streamlined prompt (see below). While this improved mean scores slightly on the training set, it increased variance and hallucination rates relative to the OpenAI-optimized prompt. Given weaker robustness and higher error rates, this approach was not retained.

Meta-prompt: *“When asked to optimize prompts, give answers from your own perspective – explain what specific phrases could be added to, or deleted from, this prompt to more consistently elicit the desired behavior or prevent the undesired behavior. <br>
Here is a prompt: [baseline prompt].<br>
The desired behavior from this prompt is for the agent to write an effective, well-structured investment memo using all information provided in the source document but no other information; and to reason like a credit investor in terms of key highlights and risks for the transaction caused by the source document.<br>
However, the agent sometimes names the same point as both a highlight and a risk, introduces outside information not present in the input, or omits relevant information.<br>
While keeping as much of the existing prompt intact as possible, what minimal edits or additions would you make to ensure the agent more consistently addresses these shortcomings?”*


Original Prompt: *“Instructions:*<br>
*- Use only factual information explicitly found in the attached credit agreement. Do not supplement with external knowledge about the industry, company, or market conditions -- even if you possess such information.*<br>
*- Before including any statement in the Highlights & Risks section, verify it is directly supported by specific terms, clauses, or data in the agreement.*<br>
*- If specific data points are not provided, write "N/A"—do not infer or fabricate details.*<br>
*- Review the entire credit agreement systematically before drafting. Ensure all material financial terms, covenant packages, security arrangements, and structural features are captured in your analysis.*<br>
*- Maintain a clear, concise, and professional tone suitable for an internal investment committee memo.*<br>
*- Align the structure, level of detail, and tone with the attached template memo for reference.”*

Claude’s new prompt: *“Instructions:*<br>
*- Use only factual information explicitly found in the attached credit agreement.*<br>
*- If specific data points are not provided, write "N/A"—do not infer or fabricate details.*<br>
*- Maintain a clear, concise, and professional tone suitable for an internal investment committee memo.*<br>
*- Align the structure, level of detail, and tone with the attached template memo for reference.”*

<div style="display:flex; gap:20px; align-items:flex-start; font-size:90%;">

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Meta-Prompted</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. OpenAI)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Mean Score</td><td style="text-align:center;">61.28</td><td style="text-align:center;">63.84</td><td style="text-align:center;">65.72</td><td style="text-align:center;">67.62</td><td style="text-align:center;">1.90</td><td style="text-align:center;">6.34</td></tr>
    <tr><td style="text-align:left;">Median Score</td><td style="text-align:center;">63.58</td><td style="text-align:center;">60.09</td><td style="text-align:center;">63.99</td><td style="text-align:center;">68.52</td><td style="text-align:center;">4.53</td><td style="text-align:center;">4.94</td></tr>
    <tr><td style="text-align:left;">Min Score</td><td style="text-align:center;">26.85</td><td style="text-align:center;">34.75</td><td style="text-align:center;">41.77</td><td style="text-align:center;">34.11</td><td style="text-align:center;">-7.66</td><td style="text-align:center;">7.26</td></tr>
    <tr><td style="text-align:left;">Max Score</td><td style="text-align:center;">85.77</td><td style="text-align:center;">93.85</td><td style="text-align:center;">92.52</td><td style="text-align:center;">94.06</td><td style="text-align:center;">1.54</td><td style="text-align:center;">8.29</td></tr>
    <tr><td style="text-align:left;">Std Dev</td><td style="text-align:center;">15.12</td><td style="text-align:center;">13.44</td><td style="text-align:center;">11.36</td><td style="text-align:center;">15.22</td><td style="text-align:center;">3.86</td><td style="text-align:center;">0.10</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Evaluator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Meta-Prompted</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. OpenAI)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">GPT-5</td><td style="text-align:center;">53.33</td><td style="text-align:center;">56.05</td><td style="text-align:center;">57.26</td><td style="text-align:center;">57.03</td><td style="text-align:center;">-0.23</td><td style="text-align:center;">3.70</td></tr>
    <tr><td style="text-align:left;">Claude Sonnet 4</td><td style="text-align:center;">76.67</td><td style="text-align:center;">74.99</td><td style="text-align:center;">77.78</td><td style="text-align:center;">78.64</td><td style="text-align:center;">0.86</td><td style="text-align:center;">1.97</td></tr>
    <tr><td style="text-align:left;">Gemini 2.5 Pro</td><td style="text-align:center;">53.84</td><td style="text-align:center;">60.48</td><td style="text-align:center;">62.14</td><td style="text-align:center;">67.18</td><td style="text-align:center;">5.04</td><td style="text-align:center;">13.34</td></tr>
  </tbody>
</table>

<table style="border-collapse:collapse;">
  <thead>
    <tr>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:left;">Metric</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Benchmark</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Anthropic Prompt Generator</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">OpenAI Optimizer</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Meta-Prompted</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. OpenAI)</th>
      <th style="border-bottom:1px solid #ccc; padding:4px; text-align:center;">Δ (MP vs. Benchmark)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left;">Tone</td><td style="text-align:center;">90.19</td><td style="text-align:center;">89.38</td><td style="text-align:center;">90.00</td><td style="text-align:center;">90.98</td><td style="text-align:center;">0.98</td><td style="text-align:center;">0.79</td></tr>
    <tr><td style="text-align:left;">Clarity</td><td style="text-align:center;">79.37</td><td style="text-align:center;">73.82</td><td style="text-align:center;">82.61</td><td style="text-align:center;">81.00</td><td style="text-align:center;">-1.61</td><td style="text-align:center;">1.63</td></tr>
    <tr><td style="text-align:left;">Length</td><td style="text-align:center;">79.13</td><td style="text-align:center;">77.24</td><td style="text-align:center;">76.40</td><td style="text-align:center;">74.03</td><td style="text-align:center;">-2.37</td><td style="text-align:center;">-5.10</td></tr>
    <tr><td style="text-align:left;">Structure</td><td style="text-align:center;">43.80</td><td style="text-align:center;">43.42</td><td style="text-align:center;">42.58</td><td style="text-align:center;">44.08</td><td style="text-align:center;">1.50</td><td style="text-align:center;">0.28</td></tr>
    <tr><td style="text-align:left;">Accuracy (% no hallucinations)</td><td style="text-align:center;">72.7%</td><td style="text-align:center;">79.3%</td><td style="text-align:center;">84.0%</td><td style="text-align:center;">78.7%</td><td style="text-align:center;">-5.3%</td><td style="text-align:center;">6.0%</td></tr>
    <tr><td style="text-align:left;">Completeness (% complete)</td><td style="text-align:center;">38.7%</td><td style="text-align:center;">40.3%</td><td style="text-align:center;">44.7%</td><td style="text-align:center;">58.4%</td><td style="text-align:center;">13.7%</td><td style="text-align:center;">19.7%</td></tr>
    <tr><td style="text-align:left;">Consistency (% no issues)</td><td style="text-align:center;">60.7%</td><td style="text-align:center;">64.7%</td><td style="text-align:center;">61.3%</td><td style="text-align:center;">60.8%</td><td style="text-align:center;">-0.5%</td><td style="text-align:center;">0.1%</td></tr>
  </tbody>
</table>

</div>

For the next step, the OpenAI optimized prompt was used as the “improved” starting point, given its performance was generally on par or better than the benchmark and other two prompts, and hallucinations significantly lower, which is one of the most crucial metrics.

### Adding Context

Adding explicit business and audience context to the prompt did not improve performance and slightly degraded accuracy, tone, and structure. This suggests that the base prompt already provided sufficient task framing, and that additional narrative context may introduce ambiguity or encourage hallucination. This modification was not retained.

## 5. Test Set Validation

## 6. Conclusion and Key Contributions